# 0. Extract: KWCS 원시자료 → 표준화 CSV

| 항목 | 내용 |
|------|------|
| **입력** | `Data_kosha/근로환경조사/` KWCS 2~7차 원시 CSV (KOSHA 공식 배포) |
| **출력** | `output/pre_output/kwcs_pooled_raw.csv` |
| **처리** | 변수명 표준화 + 결측코드 → NaN + 풀링 |
| **재현성** | 결정적(deterministic) — 동일 입력 → 동일 출력 |

In [6]:
# ══════════════════════════════════════════════════════════════
# Cell 1 — 환경 설정 + Drive write 권한 검증 (Colab/Local portable)
# ══════════════════════════════════════════════════════════════
import os, sys, time
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/완석_구글자료/연구자료/20260313_kosha'
    ENV = 'Colab'
except ImportError:
    BASE = os.path.expanduser(
        '~/Library/CloudStorage/GoogleDrive-y3korea@gmail.com/'
        '내 드라이브/완석_구글자료/연구자료/20260313_kosha')
    ENV = 'Local'
assert os.path.exists(BASE), f'BASE not found: {BASE}'
print(f'[ENV] {ENV}')
print(f'[BASE] {BASE}')

import pandas as pd, numpy as np, hashlib

DATA_DIR = os.path.join(BASE, 'Data_kosha', '근로환경조사')
OUT_DIR  = os.path.join(BASE, 'Code_kosha', '1_code', 'output', 'pooled_raw')
os.makedirs(OUT_DIR, exist_ok=True)

# ── Sanity checks (실패 시 즉시 멈춤 + 명확한 에러) ──
assert os.path.isdir(DATA_DIR), f'❌ DATA_DIR NOT FOUND: {DATA_DIR}'
assert os.path.isdir(OUT_DIR),  f'❌ OUT_DIR NOT FOUND: {OUT_DIR}'
print(f'[DATA_DIR ✓] {DATA_DIR}')
print(f'[OUT_DIR  ✓] {OUT_DIR}')

# ── Drive WRITE 권한 smoke test (가장 흔한 Colab 실패 원인 사전 차단) ──
test_path = os.path.join(OUT_DIR, '_write_test.tmp')
try:
    t0 = time.time()
    with open(test_path, 'w') as f:
        f.write('write_test_ok')
    # Read back to verify
    with open(test_path) as f:
        assert f.read() == 'write_test_ok'
    os.remove(test_path)
    print(f'[WRITE TEST ✓] OUT_DIR is writable ({(time.time()-t0)*1000:.0f} ms)')
except Exception as e:
    print(f'[WRITE TEST ✗] {type(e).__name__}: {e}')
    print(f'  → OUT_DIR={OUT_DIR}')
    print(f'  → 이 단계가 실패하면 to_csv도 실패합니다. Drive mount/권한 확인 필요.')
    raise
print('환경 설정 완료')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[ENV] Colab
[BASE] /content/drive/MyDrive/완석_구글자료/연구자료/20260313_kosha
[DATA_DIR ✓] /content/drive/MyDrive/완석_구글자료/연구자료/20260313_kosha/Data_kosha/근로환경조사
[OUT_DIR  ✓] /content/drive/MyDrive/완석_구글자료/연구자료/20260313_kosha/Code_kosha/1_code/output/pooled_raw
[WRITE TEST ✓] OUT_DIR is writable (20 ms)
환경 설정 완료


In [7]:
# ══════════════════════════════════════════════════════════════
# Cell 2 — 추출 + 표준화 + 결측 처리 + 풀링
# ══════════════════════════════════════════════════════════════
#
# ┌──────────────────────────────────────────────────────────┐
# │ 설계 결정 D1. Wave 1 (2006) 제외                         │
# │ heal_prob3(하지 MSD), hazard_psy2(작업강도) 미수집으로    │
# │ 3부위 MSD 결과변수 + JDC-S 모델 완전 구성 불가.          │
# │ STROBE flow에 제외 사유 명시.                            │
# └──────────────────────────────────────────────────────────┘
#
# ┌──────────────────────────────────────────────────────────┐
# │ 설계 결정 D2. 변수 매핑 (Table S1 기반)                   │
# │ KWCS는 EWCS를 모델로 하나 웨이브별 변수명 상이:          │
# │   연령: W2,6='AGE' / W3-5='TAGE' / W7='age'             │
# │   성별: W2-6='TSEX' / W7='gender'                       │
# │   산업: W2-6='IND'  / W7='ind'                           │
# │   가중치: W2-6='wt' / W7='wt3' (최종 보정 가중치)        │
# │ 각 차수 코드북(KOSHA 공식 배포)과 대조하여 확정.          │
# └──────────────────────────────────────────────────────────┘
#
# ┌──────────────────────────────────────────────────────────┐
# │ 설계 결정 D3. 결측 코딩 체계                              │
# │                                                          │
# │ 변수유형           유효범위  결측코드                      │
# │ ──────────────── ──────── ──────────────────────         │
# │ MSD (heal_prob)   1,2      8=모름, 9=무응답               │
# │ 빈도척도(hazard)   1-7      8=모름, 9=무응답               │
# │   (1=항상~7=전혀없음; 7은 유효한 빈도 응답)               │
# │ 자율성(decla)      1,2      8=모름, 9=무응답               │
# │ 사회적지지(wsit)   1-5      7=해당없음, 8=모름, 9=무응답   │
# │   (7은 자영업/1인사업장 = 구조적 결측)                    │
# │ 일-생활균형(wbal)  1-4      8=모름, 9=무응답               │
# │ 교육(edu)          1-7      8=모름, 9=무응답               │
# │ 고용형태(emp)      1-5      8=무응답                      │
# │ 근무시간(whrs)     연속     888=모름, 999=무응답           │
# │                                                          │
# │ 핵심 처리:                                               │
# │ hazard 변수의 7(전혀없음)은 유효한 빈도 응답이나,         │
# │ "해당 작업 미수행"으로 해석하여 결측 처리.                │
# │ 근거: 노출 수준 측정값 vs 노출 부재는 질적으로 다르며,    │
# │ EWCS 분석 매뉴얼(Eurofound, 2017)과 동일 처리.          │
# │ wsituation 7(해당없음)은 자영업/1인사업장으로             │
# │ 동료/상사가 없는 구조적 결측.                            │
# └──────────────────────────────────────────────────────────┘

RAW = {
    2: ('2차_근로환경/[CSV]2nd_KWCS_kor/★원시자료_2차 근로환경조사_제공용_220207.csv', 2010),
    3: ('3차_근로환경/[CSV]3rd_KWCS_kor/★원시자료_3차 근로환경조사_제공용_22207.csv',  2011),
    4: ('4차_근로환경/[CSV]4th_KWCS_kor/★원시자료_4차 근로환경조사_제공용_220126.csv', 2014),
    5: ('5차_근로환경/[CSV]5th_KWCS_kor/★원시자료_5차 근로환경조사_제공용_22207.csv',  2017),
    6: ('6차_근로환경/[CSV]6th_KWCS_kor/★원시자료_6차 근로환경조사_제공용_220210.csv', 2020),
    7: ('7차_근로환경/[CSV] 7th KWCS_kor/[데이터] 2023년 제7차 근로환경조사.dat',      2023),
}

COMMON = {
    'heal_prob1':'msd_back', 'heal_prob2':'msd_upper', 'heal_prob3':'msd_lower',
    'hazard_psy1':'psy_demand', 'hazard_psy2':'int_speed',
    'decla1':'auto1', 'decla2':'auto2', 'decla3':'auto3',
    'wsituation1':'sup_col', 'wsituation2':'sup_mgr',
    'hazard_erg1':'erg_posture', 'hazard_erg2':'erg_lift_ppl',
    'hazard_erg3':'erg_heavy', 'hazard_erg4':'erg_stand', 'hazard_erg6':'erg_repeat',
    'wbalance':'wlb', 'edu':'edu', 'emp_type':'emp_type', 'wtime_week':'whrs', 'wt':'wt',
}
VMAP = {
    2: {**COMMON, 'TSEX':'gender', 'AGE':'age_raw',  'IND':'industry'},
    3: {**COMMON, 'TSEX':'gender', 'TAGE':'age_raw', 'IND':'industry'},
    4: {**COMMON, 'TSEX':'gender', 'TAGE':'age_raw', 'IND':'industry'},
    5: {**COMMON, 'TSEX':'gender', 'TAGE':'age_raw', 'IND':'industry'},
    6: {**COMMON, 'TSEX':'gender', 'AGE':'age_raw',  'IND':'industry'},
    7: {**{k:v for k,v in COMMON.items() if k!='wt'},
        'gender':'gender', 'age':'age_raw', 'ind':'industry', 'wt3':'wt'},
}
OUT_COLS = ['wave','year',
    'msd_back','msd_upper','msd_lower',
    'psy_demand','auto1','auto2','auto3','sup_col','sup_mgr','int_speed',
    'erg_posture','erg_lift_ppl','erg_heavy','erg_stand','erg_repeat',
    'wlb','gender','age_raw','edu','emp_type','industry','whrs','wt']


def load_and_clean(w):
    fp = os.path.join(DATA_DIR, RAW[w][0])
    vmap = VMAP[w]
    kw = dict(usecols=list(vmap.keys()), low_memory=False)
    if w == 7:
        kw.update(sep='\t', encoding='utf-8', encoding_errors='replace')
    else:
        kw['encoding'] = 'utf-8-sig'

    df_raw = pd.read_csv(fp, **kw).rename(columns=vmap)
    for c in df_raw.columns:
        df_raw[c] = pd.to_numeric(df_raw[c], errors='coerce')

    N_raw = len(df_raw)

    # ── 결측 코드 변환 전 원시 분포 기록 (검증용) ──
    recode_log = {}
    for v in ['msd_back','msd_upper','msd_lower']:
        invalid = (~df_raw[v].isin([1,2]) & df_raw[v].notna()).sum()
        recode_log[v] = {'invalid_codes': int(invalid), 'rule': '1,2 only'}
        df_raw.loc[~df_raw[v].isin([1,2]), v] = np.nan

    for v in ['psy_demand','int_speed']:
        n7 = (df_raw[v]==7).sum()
        n89 = ((df_raw[v]>=8) & df_raw[v].notna()).sum()
        recode_log[v] = {'val_7_NA': int(n7), 'val_8_9_DK_Ref': int(n89), 'rule': '1-6 valid, 7+=NaN'}
        df_raw.loc[df_raw[v] >= 7, v] = np.nan

    for v in ['auto1','auto2','auto3']:
        invalid = (~df_raw[v].isin([1,2]) & df_raw[v].notna()).sum()
        recode_log[v] = {'invalid_codes': int(invalid), 'rule': '1,2 only'}
        df_raw.loc[~df_raw[v].isin([1,2]), v] = np.nan

    for v in ['sup_col','sup_mgr']:
        n7 = (df_raw[v]==7).sum()
        n89 = ((df_raw[v]>=8) & df_raw[v].notna()).sum()
        recode_log[v] = {'val_7_NA_structural': int(n7), 'val_8_9_DK_Ref': int(n89), 'rule': '1-5 valid, 7+=NaN'}
        df_raw.loc[df_raw[v] >= 7, v] = np.nan

    for v in ['erg_posture','erg_lift_ppl','erg_heavy','erg_stand','erg_repeat']:
        n7 = (df_raw[v]==7).sum()
        n89 = ((df_raw[v]>=8) & df_raw[v].notna()).sum()
        recode_log[v] = {'val_7_Never': int(n7), 'val_8_9_DK_Ref': int(n89), 'rule': '1-6 valid, 7+=NaN'}
        df_raw.loc[df_raw[v] >= 7, v] = np.nan

    for v, thresh in [('wlb',8),('edu',8),('emp_type',8)]:
        invalid = ((df_raw[v]>=thresh) & df_raw[v].notna()).sum()
        recode_log[v] = {'invalid_codes': int(invalid), 'rule': f'<{thresh} valid'}
        df_raw.loc[df_raw[v] >= thresh, v] = np.nan

    invalid_whrs = ((df_raw['whrs']>=888) & df_raw['whrs'].notna()).sum()
    recode_log['whrs'] = {'val_888_999': int(invalid_whrs), 'rule': '<888 valid'}
    df_raw.loc[df_raw['whrs'] >= 888, 'whrs'] = np.nan

    df_raw['wave'], df_raw['year'] = w, RAW[w][1]
    return df_raw[OUT_COLS], N_raw, recode_log


# ── 풀링 실행 ──
frames = []
all_logs = {}
for w in sorted(RAW):
    d, n_raw, log = load_and_clean(w)
    frames.append(d)
    all_logs[w] = log
    print(f'W{w} ({RAW[w][1]}): N_raw={n_raw:,} → N_valid_cols={len(d):,}')

df = pd.concat(frames, ignore_index=True)
df = df.sort_values(['wave','age_raw','gender'], na_position='last').reset_index(drop=True)

# ── 저장 (verbose, 단계별 명시) ──
out = os.path.join(OUT_DIR, 'kwcs_pooled_raw.csv')
print(f'\n[저장 시작] {out}')
print(f'  메모리상 DataFrame: {len(df):,} rows × {len(df.columns)} cols')

try:
    # 1. CSV string 생성 (메모리상)
    csv_str = df.to_csv(index=False)
    print(f'  ✓ CSV string 생성: {len(csv_str)/1024/1024:.1f} MB')

    # 2. 디스크에 쓰기 (utf-8-sig BOM 포함)
    with open(out, 'w', encoding='utf-8-sig') as f:
        f.write(csv_str)
    print(f'  ✓ 디스크 쓰기 완료')

    # 3. 즉시 검증 — 파일 실제 존재 + size 일치
    assert os.path.exists(out), f'WRITE FAILED: file not present after write'
    actual_sz = os.path.getsize(out)
    assert actual_sz > 0, f'WRITE FAILED: file size is 0'
    print(f'  ✓ 검증: 파일 존재 + size {actual_sz/1024/1024:.1f} MB')

    # 4. mtime 확인
    import datetime
    mt = datetime.datetime.fromtimestamp(os.path.getmtime(out))
    print(f'  ✓ Modified: {mt}')

    # 5. SHA-256 hash (재현성)
    sha = hashlib.sha256(csv_str.encode('utf-8-sig')).hexdigest()[:16]
    print(f'  ✓ SHA-256: {sha}')

except Exception as e:
    import traceback
    print(f'  ❌ 저장 실패: {type(e).__name__}: {e}')
    print(f'  {"="*60}')
    traceback.print_exc()
    raise

print(f'\n풀링 완료: N = {len(df):,}')
print(f'SHA-256: {sha}')
print(f'저장: {out}')

W2 (2010): N_raw=10,019 → N_valid_cols=10,019
W3 (2011): N_raw=50,032 → N_valid_cols=50,032
W4 (2014): N_raw=50,007 → N_valid_cols=50,007
W5 (2017): N_raw=50,205 → N_valid_cols=50,205
W6 (2020): N_raw=50,538 → N_valid_cols=50,538
W7 (2023): N_raw=50,195 → N_valid_cols=50,195

[저장 시작] /content/drive/MyDrive/완석_구글자료/연구자료/20260313_kosha/Code_kosha/1_code/output/pooled_raw/kwcs_pooled_raw.csv
  메모리상 DataFrame: 260,996 rows × 25 cols
  ✓ CSV string 생성: 23.6 MB
  ✓ 디스크 쓰기 완료
  ✓ 검증: 파일 존재 + size 23.6 MB
  ✓ Modified: 2026-05-10 07:42:55
  ✓ SHA-256: 191facd4cef0dc5e

풀링 완료: N = 260,996
SHA-256: 191facd4cef0dc5e
저장: /content/drive/MyDrive/완석_구글자료/연구자료/20260313_kosha/Code_kosha/1_code/output/pooled_raw/kwcs_pooled_raw.csv


In [8]:
# ══════════════════════════════════════════════════════════════
# Cell 3 — 검증 리포트 (Methods 섹션 근거 자료)
# ══════════════════════════════════════════════════════════════
#
# 이 셀의 출력은 논문 Methods + Supplementary Table S1-S3의
# 근거 자료로 직접 사용 가능합니다.
# ══════════════════════════════════════════════════════════════

YEARS = {2:2010, 3:2011, 4:2014, 5:2017, 6:2020, 7:2023}

# ── Table S1: 웨이브별 결측 코드 변환 상세 ──
print('='*70)
print('Table S1. Missing Code Recoding by Wave')
print('='*70)
print('(7=Not applicable / Never, 8=Don\'t know, 9=Refused → NaN)')
print()

for w in sorted(all_logs):
    print(f'--- Wave {w} ({YEARS[w]}) ---')
    log = all_logs[w]
    for var, info in log.items():
        parts = [f'{k}={v}' for k,v in info.items() if k != 'rule']
        print(f'  {var:20s} [{info["rule"]}] {"  ".join(parts)}')
    print()


# ── Table S2: 변수별 결측률 (웨이브 × 변수) ──
print('='*70)
print('Table S2. Missing Rate (%) by Wave and Variable (after recoding)')
print('='*70)

analysis_vars = [
    'msd_back','msd_upper','msd_lower',
    'psy_demand','auto1','auto2','auto3','sup_col','sup_mgr','int_speed',
    'erg_posture','erg_lift_ppl','erg_heavy','erg_stand','erg_repeat',
    'wlb','edu','emp_type','whrs'
]

header = f'{"Variable":20s}' + ''.join([f'  W{w}({YEARS[w]})' for w in sorted(YEARS)]) + '    Total'
print(header)
print('-'*len(header))

for v in analysis_vars:
    row = f'{v:20s}'
    for w in sorted(YEARS):
        sub = df[df['wave']==w]
        pct = sub[v].isna().mean() * 100
        row += f'  {pct:>8.1f}%'
    total_pct = df[v].isna().mean() * 100
    row += f'  {total_pct:>8.1f}%'
    print(row)

print()


# ── Table S3: 주요 변수 유효값 분포 (풀링 기준) ──
print('='*70)
print('Table S3. Valid Value Distribution (Pooled, after recoding)')
print('='*70)

for v in ['msd_back','psy_demand','auto1','sup_col','erg_posture','erg_lift_ppl','wlb','edu','emp_type']:
    vc = df[v].dropna().astype(int).value_counts().sort_index()
    total_valid = vc.sum()
    total_miss = df[v].isna().sum()
    dist = ', '.join([f'{k}:{v:,}({v/total_valid*100:.1f}%)' for k,v in vc.items()])
    print(f'\n{v} (valid={total_valid:,}, missing={total_miss:,}, miss%={total_miss/len(df)*100:.1f}%)')
    print(f'  {dist}')

print()


# ── 결측 구조 분석: 논문 리뷰어 대응 핵심 ──
print('='*70)
print('Missing Pattern Analysis (for Reviewer Response)')
print('='*70)

# 어떤 변수 때문에 주로 제외되는지
print('\n1. 변수별 결측이 표본 제외에 미치는 영향 (단독 기여):')
base_valid = df.dropna(subset=['msd_back','msd_upper','msd_lower','gender','age_raw','wt'])
N_base = len(base_valid)
print(f'   기본 유효 (MSD+demographics): {N_base:,}')

for v in ['psy_demand','sup_col','sup_mgr','erg_posture','erg_lift_ppl',
          'erg_heavy','erg_stand','erg_repeat','emp_type','whrs']:
    n_lost = base_valid[v].isna().sum()
    print(f'   + {v:20s} 결측 시 추가 제외: {n_lost:>7,} ({n_lost/N_base*100:.1f}%)')

print(f'\n2. 구조적 결측(7=N/A) vs 무응답(8/9) 비중:')
for w in sorted(YEARS):
    log = all_logs[w]
    for v in ['sup_col','erg_lift_ppl']:
        info = log.get(v, {})
        na_struct = info.get('val_7_NA_structural', info.get('val_7_Never', 0))
        na_dk_ref = info.get('val_8_9_DK_Ref', 0)
        print(f'   W{w} {v:20s} 구조적={na_struct:>6,}  무응답={na_dk_ref:>5,}')

print(f'\n3. sup_col=7(해당없음)인 응답자의 sup_mgr 유효율:')
for w in sorted(YEARS):
    sub = df[df['wave']==w]
    # sup_col이 NaN이면서 sup_mgr이 유효한 비율
    sup_col_na = sub[sub['sup_col'].isna()]
    if len(sup_col_na) > 0:
        sup_mgr_valid = sup_col_na['sup_mgr'].notna().sum()
        print(f'   W{w}: sup_col 결측 {len(sup_col_na):,}명 중 sup_mgr 유효 {sup_mgr_valid:,}명 ({sup_mgr_valid/len(sup_col_na)*100:.1f}%)')

print('\n   → sup_col=NaN이더라도 sup_mgr로 support 산출 가능 (skipna=True)')
print('   → 1_preprocess에서 key_vars에 sup_col 미포함, support는 파생변수로 계산')

print(f'\n저장: {out}')
print('다음 단계: 1_preprocess.ipynb 실행')

Table S1. Missing Code Recoding by Wave
(7=Not applicable / Never, 8=Don't know, 9=Refused → NaN)

--- Wave 2 (2010) ---
  msd_back             [1,2 only] invalid_codes=0
  msd_upper            [1,2 only] invalid_codes=0
  msd_lower            [1,2 only] invalid_codes=0
  psy_demand           [1-6 valid, 7+=NaN] val_7_NA=3684  val_8_9_DK_Ref=0
  int_speed            [1-6 valid, 7+=NaN] val_7_NA=6171  val_8_9_DK_Ref=0
  auto1                [1,2 only] invalid_codes=0
  auto2                [1,2 only] invalid_codes=0
  auto3                [1,2 only] invalid_codes=0
  sup_col              [1-5 valid, 7+=NaN] val_7_NA_structural=3077  val_8_9_DK_Ref=0
  sup_mgr              [1-5 valid, 7+=NaN] val_7_NA_structural=262  val_8_9_DK_Ref=0
  erg_posture          [1-6 valid, 7+=NaN] val_7_Never=2338  val_8_9_DK_Ref=0
  erg_lift_ppl         [1-6 valid, 7+=NaN] val_7_Never=6945  val_8_9_DK_Ref=0
  erg_heavy            [1-6 valid, 7+=NaN] val_7_Never=3410  val_8_9_DK_Ref=0
  erg_stand            [

In [9]:
# ══════════════════════════════════════════════════════════════
# Cell 4 — 원천 파일 경로 + 논문 Methods 기재용 텍스트
# ══════════════════════════════════════════════════════════════

YEARS = {2:2010, 3:2011, 4:2014, 5:2017, 6:2020, 7:2023}

print('='*70)
print('SOURCE FILE DOCUMENTATION')
print('='*70)
print(f'''
[Survey]
  Korean Working Conditions Survey (KWCS, 근로환경조사)
  Agency: Korea Occupational Safety and Health Agency (KOSHA)
  Design: Nationally representative repeated cross-sectional survey
          of employed persons aged ≥15 years (modelled after EWCS)
  Method: Multi-stage stratified probability sampling, face-to-face interviews

[Source Files — 6 waves pooled]''')
for w in sorted(RAW):
    fp = os.path.join(DATA_DIR, RAW[w][0])
    sub = df[df['wave']==w]
    print(f'  W{w} ({YEARS[w]}): N={len(sub):,}')
    print(f'    File: {RAW[w][0]}')
    print(f'    Path: Data_kosha/근로환경조사/{RAW[w][0]}')

print(f'''
[Excluded]
  Wave 1 (2006): heal_prob3(하지 MSD), hazard_psy2(작업강도) 미수집
                  → 3부위 MSD + JDC-S 모델 완전 구성 불가

[Variable Mapping — Wave-specific]
  연령: W2,6='AGE' / W3-5='TAGE' / W7='age'    → age_raw
  성별: W2-6='TSEX' / W7='gender'               → gender
  산업: W2-6='IND'  / W7='ind'                  → industry
  가중치: W2-6='wt' / W7='wt3'                  → wt

[Standardized Variables — {len(OUT_COLS)} columns]
  Outcomes (3)   : msd_back, msd_upper, msd_lower (heal_prob1-3)
  JDC-S (7)      : psy_demand, auto1-3, sup_col, sup_mgr, int_speed
  Ergonomic (5)  : erg_posture, erg_lift_ppl, erg_heavy, erg_stand, erg_repeat
  Covariates (6) : wlb, gender, age_raw, edu, emp_type, whrs
  Meta (3)       : wave, year, industry, wt

[Missing Code Rules]
  MSD (heal_prob)  : 1,2 valid → 8,9=NaN
  Frequency scales : 1-6 valid → 7(=never/NA)=NaN, 8,9=NaN
  Autonomy (decla) : 1,2 valid → 8,9=NaN
  Support (wsit)   : 1-5 valid → 7(=structural missing)=NaN, 8,9=NaN
  Working hours    : continuous → 888,999=NaN

[Output]
  File: output/pooled_raw/kwcs_pooled_raw.csv
  N   : {len(df):,} rows × {len(OUT_COLS)} columns
  SHA : {sha}
''')

print('='*70)
print('FOR PAPER — Methods §2.1 Study Design and Data Source')
print('='*70)

# Wave-specific N for paper
wave_ns = {w: len(df[df['wave']==w]) for w in sorted(YEARS)}
total_n = sum(wave_ns.values())

print(f'''
  "We pooled data from six waves (2nd–7th, 2010–2023) of the
  Korean Working Conditions Survey (KWCS), a nationally
  representative repeated cross-sectional survey of employed
  persons aged ≥15 years conducted by the Korea Occupational
  Safety and Health Agency (KOSHA). The KWCS uses multi-stage
  stratified probability sampling with face-to-face interviews
  and is modelled after the European Working Conditions Survey
  (EWCS). The 1st wave (2006) was excluded because lower limb
  MSD and work intensity items were unavailable.

  Wave-specific sample sizes were: W2 (2010, N={wave_ns[2]:,}),
  W3 (2011, N={wave_ns[3]:,}), W4 (2014, N={wave_ns[4]:,}),
  W5 (2017, N={wave_ns[5]:,}), W6 (2020, N={wave_ns[6]:,}),
  and W7 (2023, N={wave_ns[7]:,}), yielding a total pooled
  sample of {total_n:,} respondents.

  Variables were standardised across waves using KOSHA codebooks
  (variable naming differs across waves). Missing codes (8='don't
  know', 9='refused') were recoded to missing. For frequency
  scales, code 7 ('never') was treated as missing, representing
  non-exposure rather than a measurable exposure level, consistent
  with EWCS analysis guidelines (Eurofound, 2017)."
''')
print('다음 단계: 1_preprocess.ipynb 실행')

SOURCE FILE DOCUMENTATION

[Survey]
  Korean Working Conditions Survey (KWCS, 근로환경조사)
  Agency: Korea Occupational Safety and Health Agency (KOSHA)
  Design: Nationally representative repeated cross-sectional survey
          of employed persons aged ≥15 years (modelled after EWCS)
  Method: Multi-stage stratified probability sampling, face-to-face interviews

[Source Files — 6 waves pooled]
  W2 (2010): N=10,019
    File: 2차_근로환경/[CSV]2nd_KWCS_kor/★원시자료_2차 근로환경조사_제공용_220207.csv
    Path: Data_kosha/근로환경조사/2차_근로환경/[CSV]2nd_KWCS_kor/★원시자료_2차 근로환경조사_제공용_220207.csv
  W3 (2011): N=50,032
    File: 3차_근로환경/[CSV]3rd_KWCS_kor/★원시자료_3차 근로환경조사_제공용_22207.csv
    Path: Data_kosha/근로환경조사/3차_근로환경/[CSV]3rd_KWCS_kor/★원시자료_3차 근로환경조사_제공용_22207.csv
  W4 (2014): N=50,007
    File: 4차_근로환경/[CSV]4th_KWCS_kor/★원시자료_4차 근로환경조사_제공용_220126.csv
    Path: Data_kosha/근로환경조사/4차_근로환경/[CSV]4th_KWCS_kor/★원시자료_4차 근로환경조사_제공용_220126.csv
  W5 (2017): N=50,205
    File: 5차_근로환경/[CSV]5th_KWCS_kor/★원시자료_5차 근로환경조사_제공용_22207.c

In [10]:
# ══════════════════════════════════════════════════════════════
# DIAGNOSTIC — 저장 결과 즉시 확인 (덮어쓰기 성공 여부)
# ══════════════════════════════════════════════════════════════
import os, datetime
print('='*60)
print('SAVE DIAGNOSTIC')
print('='*60)

out_path = os.path.join(OUT_DIR, 'kwcs_pooled_raw.csv')
if os.path.exists(out_path):
    sz = os.path.getsize(out_path) / 1024 / 1024
    mt = datetime.datetime.fromtimestamp(os.path.getmtime(out_path))
    print(f'✅ Saved: {out_path}')
    print(f'   Size  : {sz:.1f} MB')
    print(f'   Modified: {mt}')
    if (datetime.datetime.now() - mt).total_seconds() < 600:
        print(f'   ✓ 방금 덮어쓰기 성공 (10분 이내)')
    else:
        print(f'   ⚠ 파일이 오래됨 — 이번 실행이 덮어쓰기 실패한 것일 수 있음')
else:
    print(f'❌ NOT FOUND: {out_path}')
    print(f'   OUT_DIR exists? {os.path.isdir(OUT_DIR)}')
    print(f'   OUT_DIR contents: {os.listdir(OUT_DIR) if os.path.isdir(OUT_DIR) else "(missing)"}')

SAVE DIAGNOSTIC
✅ Saved: /content/drive/MyDrive/완석_구글자료/연구자료/20260313_kosha/Code_kosha/1_code/output/pooled_raw/kwcs_pooled_raw.csv
   Size  : 23.6 MB
   Modified: 2026-05-10 07:42:55
   ✓ 방금 덮어쓰기 성공 (10분 이내)
